# 03 — Carga do BCI Competition IV 2a (MOABB/braindecode)

Prepara os dados para ML. Pré-requisito: rode antes o [`02_eda_bci_iv_2a.ipynb`](02_eda_bci_iv_2a.ipynb) e veja a ficha técnica em [`../README.md`](../README.md).

Dataset **BCI IV 2a** (`BNCI2014_001`): 9 sujeitos saudáveis, imagética motora de 4 classes (mão esq., mão dir., pés, língua), 22 canais EEG, 250 Hz.

O MOABB baixa o dataset e o braindecode entrega janelas já como `Dataset` PyTorch, prontas para `DataLoader`. Objetivo deste notebook: baixar → pré-processar → janelar → inspecionar → montar os `DataLoader`.

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader

from braindecode.datasets import MOABBDataset
from braindecode.preprocessing import (
    Preprocessor,
    preprocess,
    create_windows_from_events,
    exponential_moving_standardize,
)

DATASET_NAME = "BNCI2014_001"  # BCI Competition IV 2a
SUBJECT_IDS = list(range(1, 10))
CLASS_MAP = {"left_hand": 0, "right_hand": 1, "feet": 2, "tongue": 3}

# banda mu/beta + padronização online
LOW_CUT_HZ = 4.0
HIGH_CUT_HZ = 38.0
FACTOR_NEW = 1e-3
INIT_BLOCK_SIZE = 1000
TRIAL_START_OFFSET_SECONDS = -0.5  # 0,5 s antes do cue

## 1. Download

Na primeira execução o MOABB baixa os arquivos para o cache local (`~/mne_data`). Requer internet.

In [ ]:
dataset = MOABBDataset(dataset_name=DATASET_NAME, subject_ids=SUBJECT_IDS)
print(f"{len(dataset.datasets)} gravações carregadas.")
dataset.description.head()

## 2. Pré-processamento

Cada passo é explícito e reprodutível (nada de caixa-preta): seleciona só EEG, converte V→µV, aplica passa-banda 4–38 Hz e padroniza.

In [ ]:
preprocessors = [
    Preprocessor("pick_types", eeg=True, eog=False, stim=False),
    Preprocessor(lambda data: np.multiply(data, 1e6)),  # V -> uV
    Preprocessor("filter", l_freq=LOW_CUT_HZ, h_freq=HIGH_CUT_HZ),
    Preprocessor(
        exponential_moving_standardize,
        factor_new=FACTOR_NEW,
        init_block_size=INIT_BLOCK_SIZE,
    ),
]
preprocess(dataset, preprocessors)
print("Pré-processamento aplicado.")

## 3. Janelamento (trials → janelas)

`create_windows_from_events` usa os eventos/cue de cada trial para recortar as janelas rotuladas.

In [ ]:
sfreq = dataset.datasets[0].raw.info["sfreq"]
assert all(ds.raw.info["sfreq"] == sfreq for ds in dataset.datasets), "sfreq inconsistente"
trial_start_offset_samples = int(TRIAL_START_OFFSET_SECONDS * sfreq)

windows_dataset = create_windows_from_events(
    dataset,
    trial_start_offset_samples=trial_start_offset_samples,
    trial_stop_offset_samples=0,
    preload=True,
    mapping=CLASS_MAP,
)
print(f"{len(windows_dataset)} janelas no total.")

## 4. Inspeção

Confere shape de uma janela, o mapeamento de classes e a contagem por classe.

In [ ]:
X, y, _ = windows_dataset[0]
print("Shape de uma janela (canais, amostras):", X.shape)
print("Rótulo:", y)

import pandas as pd
# Rótulos direto dos metadados (rápido, sem carregar o sinal).
labels = np.concatenate([w.metadata["target"].to_numpy() for w in windows_dataset.datasets])
name_by_target = {v: k for k, v in CLASS_MAP.items()}
counts = pd.Series(labels).value_counts().sort_index()
counts.index = [name_by_target[i] for i in counts.index]
print("\nContagem por classe:")
print(counts)

## 5. Split e DataLoaders

O protocolo padrão do 2a é **within-subject**: sessão de treino → treino, sessão de avaliação → teste.

> ⚠️ O nome das sessões varia conforme a versão do MOABB (`0train`/`1test` nas versões novas, `session_T`/`session_E` nas antigas). A célula abaixo **imprime as chaves reais** — confira o print antes de confiar no mapeamento treino/teste.

In [ ]:
splits_by_session = windows_dataset.split("session")
print("Sessões encontradas:", list(splits_by_session.keys()))

splits_by_subject = windows_dataset.split("subject")
print("Sujeitos encontrados:", list(splits_by_subject.keys()))

In [ ]:
# Ajuste as duas chaves abaixo conforme o print da célula anterior.
TRAIN_SESSION = "0train"  # ou "session_T"
TEST_SESSION = "1test"    # ou "session_E"

train_set = splits_by_session[TRAIN_SESSION]
test_set = splits_by_session[TEST_SESSION]

BATCH_SIZE = 64
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

xb, yb, _ = next(iter(train_loader))
print("Batch X:", xb.shape, xb.dtype)  # (batch, canais, amostras)
print("Batch y:", yb.shape, yb.dtype)
print(f"Treino: {len(train_set)} janelas | Teste: {len(test_set)} janelas")

## Próximos passos

- `train_loader`/`test_loader` já entram direto num modelo do braindecode (`EEGNetv4`, `ShallowFBCSPNet`, `Deep4Net`).
- Para estudo **cross-subject**, use `splits_by_subject` (treinar em N sujeitos, testar em outro).